# Sliced Wasserstein Gradient Descent

The **Wasserstein distance** $W_2(\mu,\nu)$ between two probability measures quantifies the minimal transport cost to morph one distribution into the other. Computing $W_2$ in high dimension is expensive ($O(n^3)$ for discrete measures of size $n$). The **sliced Wasserstein distance** sidesteps this by averaging one-dimensional Wasserstein distances over random projections:

$$
\mathrm{SW}^2(\mu,\nu) = \int_{\mathbb{S}^{d-1}} W_2^2(\theta_{\#}\mu,\, \theta_{\#}\nu)\, d\theta,
$$

where $\theta_{\#}\mu$ denotes the pushforward of $\mu$ under the linear map $x \mapsto \theta^\top x$. Each one-dimensional $W_2^2$ is computed in $O(n \log n)$ via sorting, making $\mathrm{SW}^2$ practical at scale.

## Gradient descent via random projections

Given source particles $\{x_i\}_{i=1}^n$ (empirical measure $\mu$) and fixed target particles $\{y_j\}_{j=1}^n$ (empirical measure $\nu$), we minimise $\mathrm{SW}^2(\mu,\nu)$ over the source positions by gradient descent:
$$
x_i \leftarrow x_i - \eta\, \nabla_{x_i}\, \mathrm{SW}^2(\mu,\nu).
$$

The gradient with respect to source particle $x_i$ is:
$$
\nabla_{x_i}\, \mathrm{SW}^2 = \frac{2}{K}\sum_{k=1}^{K} \bigl(\theta_k^\top x_i - \theta_k^\top y_{\sigma_k(i)}\bigr)\,\theta_k,
$$

where $\{\theta_k\}$ are $K$ random unit directions, and $\sigma_k$ is the permutation that matches the sorted projections of source to target along direction $\theta_k$. This is an unbiased Monte Carlo estimator of the true gradient.

## What this notebook demonstrates

- How to implement the sliced Wasserstein gradient and run the optimisation.
- The step-by-step evolution of source particles converging to a multi-cluster target.
- Individual particle trajectories during the flow.
- An interactive slider for exploring the optimisation path.

## Setup

We import the standard scientific stack. No specialised optimal transport library is needed: everything is built from sorting and random projections.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = False

OUT = Path("python/sliced-wasserstein")
OUT.mkdir(parents=True, exist_ok=True)

## Sliced Wasserstein gradient and distributions

The **SW gradient** for a single direction $\theta \in \mathbb{S}^{d-1}$ is computed as follows:

1. Project: $p_i = \theta^\top x_i$ and $q_j = \theta^\top y_j$.
2. Sort both sets of projections; let $\sigma$ be the sorting permutation of $p$, and $\tau$ the sorting permutation of $q$.
3. The 1D Wasserstein gradient at $x_{\sigma(i)}$ is $(p_{\sigma(i)} - q_{\tau(i)}) \theta$.

Averaging over $K$ random directions gives the Monte Carlo estimate used in each step.

**Source distribution**: $n=300$ points drawn from $\mathcal{N}(0, I_2)$ (centred Gaussian, shown in blue).

**Target distribution**: $n=300$ points from a three-cluster Gaussian mixture with well-separated means (shown in red).

In [ ]:
rng = np.random.default_rng(42)

n = 300
n_proj = 50    # random directions per step
n_iter = 500   # total gradient steps
lr = 0.02      # step size
store_every = 10  # store trajectory every 10 steps

# Source: centred Gaussian (blue)
Xs0 = rng.normal(loc=[0.0, 0.0], scale=0.6, size=(n, 2))

# Target: 3-cluster Gaussian mixture (red)
n1, n2 = n // 3, n // 3
n3 = n - n1 - n2
Xt = np.vstack([
    rng.normal(loc=[-2.0,  1.5], scale=0.35, size=(n1, 2)),
    rng.normal(loc=[ 2.0,  1.5], scale=0.35, size=(n2, 2)),
    rng.normal(loc=[ 0.0, -2.0], scale=0.35, size=(n3, 2)),
])

def sw_grad(X, Y, n_dir=50, rng_=None):
    """Monte Carlo estimate of the SW^2 gradient w.r.t. X."""
    if rng_ is None:
        rng_ = np.random.default_rng()
    G = np.zeros_like(X)
    # Sample random unit directions on S^{d-1}
    raw = rng_.standard_normal((n_dir, X.shape[1]))
    thetas = raw / np.linalg.norm(raw, axis=1, keepdims=True)
    for th in thetas:
        px = X @ th          # projections of source
        py = Y @ th          # projections of target
        ix = np.argsort(px)
        iy = np.argsort(py)
        diff = np.empty(len(X))
        diff[ix] = px[ix] - py[iy]   # matched 1D residuals
        G += diff[:, None] * th[None, :]  # accumulate gradient
    return (2.0 / n_dir) * G

print(f"Source shape: {Xs0.shape}, Target shape: {Xt.shape}")

## Running the optimisation and storing the trajectory

We run $n_{\text{iter}} = 500$ gradient steps with step size $\eta = 0.02$, using $K = 50$ random projections per step. The source particle cloud is stored every 10 iterations so we can visualise the full trajectory.

In [ ]:
Xs = Xs0.copy()
traj = [Xs.copy()]   # traj[k] = positions at iteration k*store_every

for it in range(1, n_iter + 1):
    grad = sw_grad(Xs, Xt, n_dir=n_proj, rng_=rng)
    Xs = Xs - lr * grad
    if it % store_every == 0:
        traj.append(Xs.copy())

traj = np.array(traj)  # shape (n_snapshots, n, 2)
snap_iters = [0] + list(range(store_every, n_iter + 1, store_every))
print(f"Stored {len(traj)} snapshots over {n_iter} iterations.")

## Progression snapshots

We show six snapshots of the source cloud (blue) at iterations $0, 20, 50, 100, 200, 500$, overlaid on the fixed target (red). The source progressively rearranges itself to match the three-cluster structure.

In [ ]:
snap_display = [0, 20, 50, 100, 200, 500]  # iterations to display
snap_indices = [s // store_every for s in snap_display]  # indices into traj
# Clamp to valid range
snap_indices = [min(s, len(traj) - 1) for s in snap_indices]

fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
axes = axes.ravel()

# Compute global axis limits from all snapshots
all_pts = np.vstack([traj[0], traj[-1], Xt])
pad = 0.5
xlim = (all_pts[:, 0].min() - pad, all_pts[:, 0].max() + pad)
ylim = (all_pts[:, 1].min() - pad, all_pts[:, 1].max() + pad)

for ax, idx, it in zip(axes, snap_indices, snap_display):
    Xs_snap = traj[idx]
    ax.scatter(Xt[:, 0], Xt[:, 1], s=8, alpha=0.45, color="tab:red",
               label="target" if it == 0 else None, zorder=2)
    ax.scatter(Xs_snap[:, 0], Xs_snap[:, 1], s=8, alpha=0.55, color="tab:blue",
               label="source" if it == 0 else None, zorder=3)
    ax.set_title(f"iter {it}", fontsize=11)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Add a shared legend in the first panel
axes[0].legend(loc="upper right", fontsize=9, markerscale=2)
fig.suptitle("SW gradient descent: source (blue) converging to target (red)",
             fontsize=12, y=1.01)
plt.savefig(OUT / "_snapshots.png", bbox_inches="tight")

## Particle trajectories

For a subsample of 25 source particles, we draw the full trajectory as a polyline from the initial position (blue circle) to the final position (red circle). The colour of each polyline fades from blue (start) to red (end), showing how each particle is transported during the gradient flow.

These curves illustrate the **particle flow** induced by the sliced Wasserstein gradient: particles starting near the origin disperse and cluster around the three target modes.

In [ ]:
n_sub = 25   # number of particles to trace
rng2 = np.random.default_rng(7)
sub_idx = rng2.choice(n, size=n_sub, replace=False)

fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)

# Background: full initial source (faint blue) and target (faint red)
ax.scatter(traj[0][:, 0], traj[0][:, 1], s=5, alpha=0.15, color="tab:blue", zorder=1)
ax.scatter(Xt[:, 0], Xt[:, 1], s=5, alpha=0.15, color="tab:red", zorder=1)

n_snaps = len(traj)
cmap = cm.get_cmap("coolwarm")

for i in sub_idx:
    pts = traj[:, i, :]    # (n_snaps, 2) for particle i
    # Draw segments with colour fading blue->red
    for k in range(n_snaps - 1):
        t = k / (n_snaps - 1)
        ax.plot(pts[k:k+2, 0], pts[k:k+2, 1],
                lw=1.2, alpha=0.7, color=cmap(t), zorder=2)
    # Mark start (blue) and end (red)
    ax.scatter(*pts[0], s=40, color="tab:blue", zorder=5, edgecolors="white", linewidths=0.5)
    ax.scatter(*pts[-1], s=40, color="tab:red",  zorder=5, edgecolors="white", linewidths=0.5)

# Fake legend entries
ax.scatter([], [], s=40, color="tab:blue", label="start (source)")
ax.scatter([], [], s=40, color="tab:red",  label="end (target side)")
ax.legend(fontsize=10, loc="upper right")
ax.set_aspect("equal")
ax.set_title(f"Trajectories of {n_sub} source particles during SW gradient descent",
             fontsize=11)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(OUT / "_trajectories.png", bbox_inches="tight")

## Interactive slider: exploring the optimisation

The slider below lets you scrub through all stored snapshots of the source cloud as it flows towards the target. The iteration counter shows the current optimisation step.

Observe how the source (blue) progressively splits from a single Gaussian lobe into three distinct clusters matching the target (red).

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, IntSlider

# Axis limits (same as snapshot panel)
_all = np.vstack([traj[0], traj[-1], Xt])
_pad = 0.5
_xlim = (_all[:, 0].min() - _pad, _all[:, 0].max() + _pad)
_ylim = (_all[:, 1].min() - _pad, _all[:, 1].max() + _pad)

def show_iter(snapshot_index=0):
    it = snap_iters[snapshot_index]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(Xt[:, 0], Xt[:, 1], s=10, alpha=0.5, color="tab:red",  label="target", zorder=2)
    ax.scatter(traj[snapshot_index][:, 0], traj[snapshot_index][:, 1],
               s=10, alpha=0.6, color="tab:blue", label="source", zorder=3)
    ax.set_xlim(_xlim); ax.set_ylim(_ylim)
    ax.set_aspect("equal")
    ax.set_title(f"Iteration {it} / {n_iter}")
    ax.legend(fontsize=9, loc="upper right", markerscale=2)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.tight_layout()
    plt.show()

interact(show_iter,
         snapshot_index=IntSlider(
             value=0, min=0, max=len(traj) - 1, step=1,
             description="snapshot",
             style={"description_width": "initial"},
             layout=widgets.Layout(width="500px")
         ));

## Static snapshot

This cell saves a representative figure (`snippet.png`) combining the initial/final states with the target, and is used as the gallery thumbnail. The `STATIC_SNAPSHOT` guard ensures this cell is only executed in automated rendering contexts.

In [ ]:
STATIC_SNAPSHOT = True

if STATIC_SNAPSHOT:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), constrained_layout=True)

    # Panel 1: initial source
    axes[0].scatter(traj[0][:, 0], traj[0][:, 1], s=8, alpha=0.6, color="tab:blue")
    axes[0].set_title("Source (initial)", fontsize=11)

    # Panel 2: target
    axes[1].scatter(Xt[:, 0], Xt[:, 1], s=8, alpha=0.6, color="tab:red")
    axes[1].set_title("Target", fontsize=11)

    # Panel 3: final source overlaid on target
    axes[2].scatter(Xt[:, 0], Xt[:, 1], s=6, alpha=0.4, color="tab:red",  label="target")
    axes[2].scatter(traj[-1][:, 0], traj[-1][:, 1], s=8, alpha=0.6, color="tab:blue", label="source (final)")
    axes[2].set_title("Final matching", fontsize=11)
    axes[2].legend(fontsize=8, markerscale=2)

    for ax in axes:
        ax.set_aspect("equal")
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    fig.suptitle("Sliced Wasserstein gradient descent", fontsize=13)
    fig.savefig(OUT / "snippet.png", bbox_inches="tight")

## Takeaways

- The **sliced Wasserstein distance** replaces a single expensive 2D transport problem by an average of cheap 1D problems, enabling scalable gradient flows.
- Each gradient step costs $O(K n \log n)$ (sorting dominates), with $K$ random projections, making it practical for large particle systems.
- Despite the non-convexity of the particle optimisation landscape, the gradient flow reliably disperses a unimodal source into a multi-cluster target.
- The particle trajectories reveal the **splitting mechanism**: particles near the origin fan out in three directions, guided by the averaged projection gradients.
- Increasing $K$ (projections per step) reduces gradient noise; increasing $\eta$ speeds convergence but can cause instability.

## Bibliography

- Rabin, J., Peyré, G., Delon, J., and Bernot, M. (2012). Wasserstein Barycenter and its Application to Texture Mixing. *Scale Space and Variational Methods in Computer Vision*, 435–446.
- Bonneel, N., Rabin, J., Peyré, G., and Pfister, H. (2015). Sliced and Radon Wasserstein Barycenters of Measures. *Journal of Mathematical Imaging and Vision*, 51(1), 22–45.
- Deshpande, I., Zhang, Z., and Schwing, A. (2018). Generative Modeling Using the Sliced Wasserstein Distance. *CVPR 2018*.
- Peyré, G. and Cuturi, M. (2019). *Computational Optimal Transport*. Foundations and Trends in Machine Learning, 11(5–6). [arxiv:1803.00567](https://arxiv.org/abs/1803.00567)
- Liutkus, A., Simsekli, U., Majewski, S., Durmus, A., and Stöter, F.-R. (2019). Sliced-Wasserstein Flows: Nonparametric Generative Modeling via Optimal Transport and Diffusions. *ICML 2019*.